In [0]:
lift_data = [
    (1,300),
    (2,350)
]

lift_schema = "id int , capacity_kg int"
lift_df = spark.createDataFrame(data = lift_data , schema = lift_schema)

lift_passengers_data = [
    ('Rahul',85,1),
    ('Adarsh',73,1),
    ('Riti',95,1),
    ('Viraj',80,1),
    ('Vimal',83,2),
    ('Neha',77,2),
    ('Priti',73,2),
    ('Himanshi',85,2)
]

lift_passengers_schema = "passenger_name string , weight_kg int, lift_id int"

passengers_df = spark.createDataFrame(data = lift_passengers_data , schema = lift_passengers_schema)


In [0]:
df_join = passengers_df.join(lift_df, passengers_df.lift_id == lift_df.id,"inner")
df_join = df_join.select("passenger_name","weight_kg","lift_id","capacity_kg")
df_join.display()


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

windowspec = Window.partitionBy("lift_id").orderBy("weight_kg")

df_join = df_join.withColumn("running_wt",sum("weight_kg").over(windowspec))
df_join.display()

In [0]:
df= df_join.filter(col("capacity_kg") >= col("running_wt"))
df_final_table = df.groupBy("lift_id").agg(concat_ws(",",collect_list("passenger_name")).alias("names"))
display(df_final_table)